# Fase 02 — EDA completo (100% dos dados, pré-split)

**Pergunta:** que correlações, distribuições, outliers e padrões geográficos existem — e como o dado
se distribui por zipcode, antes de qualquer decisão de split?

**Hipótese:** tamanho/qualidade dominam a formação de preço; existe estrutura espacial forte; a
representação por zipcode é desigual o suficiente para merecer investigação formal antes de fixar o
split (P1: EDA descritivo roda em 100% dos dados, só depois é que o split é decidido).

**Módulo:** `src/scripts/run_eda.py::run_eda_pipeline` (usa
`src/evaluation/geographic_coverage.py::zip_representation_summary`).

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / ".git").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)

In [ ]:
from src.scripts.run_eda import run_eda_pipeline

report = run_eda_pipeline()
{k: v for k, v in report.items() if k not in ("correlation_with_price_log", "low_volume_zipcodes")}

{'n_rows': 21612,
 'n_zipcodes': 70,
 'zip_summary_min_n': 50,
 'zip_summary_max_n': 601,
 'n_zipcodes_below_100_properties': 3}

In [3]:
import pandas as pd

corr = pd.Series(report["correlation_with_price_log"]).sort_values(key=abs, ascending=False)
corr.to_frame("correlation_with_price_log")

,correlation_with_price_log
grade,0.703661
sqft_living,0.695365
hous_val_amt,0.630817
sqft_living15,0.619358
per_bchlr,0.611401
sqft_above,0.601844
bathrooms,0.550823
bedrooms,0.351005
view,0.346534
sqft_basement,0.316957


In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 6))
corr.sort_values().plot(kind="barh", ax=ax, color=["#55A868" if v > 0 else "#C44E52" for v in corr.sort_values().values])
ax.set_title("correlacao com price_log (100% dos dados, pre-split)")
plt.tight_layout()
plt.savefig("reports/figures/02_correlation_price_log.png", dpi=110)
plt.close()
print("saved reports/figures/02_correlation_price_log.png")

saved reports/figures/02_correlation_price_log.png


In [5]:
# lat/long nao entram no report (so entram como insumo de plotagem) -> carregado do parquet que a
# fase 01 escreveu, mesma fonte que o pipeline desta fase leu.
import numpy as np

df = pd.read_parquet("data/processed/house_clean.parquet")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
sc = axes[0].scatter(df["long"], df["lat"], c=df["price_log"], cmap="viridis", s=3, alpha=0.4)
axes[0].set_title("price_log por localizacao (100% dos dados)")
plt.colorbar(sc, ax=axes[0], label="price_log")

zip_median = df.groupby("zipcode")["price_log"].median().sort_values()
axes[1].barh(zip_median.index.astype(str), zip_median.values, color="#4C72B0")
axes[1].set_title(f"mediana price_log por zipcode ({report['n_zipcodes']} zipcodes)")
axes[1].tick_params(axis="y", labelsize=4)
plt.tight_layout()
plt.savefig("reports/figures/02_geo_price_pattern.png", dpi=110)
plt.close()
razao = np.expm1(zip_median.max()) / np.expm1(zip_median.min())
print("razao caro/barato:", razao)

razao caro/barato: 8.053015822117965


In [6]:
# reports/zip_representation_summary.csv e escrito pela propria run_eda_pipeline() acima -- releitura
# do artefato persistido (side effect natural da fase), report so traz os agregados (min/max/n abaixo
# de 100), nao a tabela completa por zipcode.
zip_summary = pd.read_csv("reports/zip_representation_summary.csv")
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(zip_summary["n"], bins=20, color="#DD8452")
ax.axvline(100, color="red", linestyle="--", label="100 imoveis")
ax.set_title(f"distribuicao do n de imoveis por zipcode ({report['n_zipcodes']} zipcodes)")
ax.set_xlabel("n imoveis no zipcode")
ax.legend()
plt.tight_layout()
plt.savefig("reports/figures/02_zip_volume_distribution.png", dpi=110)
plt.close()
print(zip_summary.sort_values("n").head(5)[["zipcode", "n", "median_price"]])
print(zip_summary.sort_values("n", ascending=False).head(5)[["zipcode", "n", "median_price"]])

   zipcode    n  median_price
0    98039   50     1892500.0
1    98148   57      278000.0
2    98024   81      460000.0
3    98010  100      359999.5
4    98102  105      720000.0
    zipcode    n  median_price
69    98103  601      550000.0
68    98038  590      342000.0
67    98115  583      567000.0
66    98052  574      615000.0
65    98117  553      544000.0


## Análise

**Correlação (100% dos dados, execução real acima):** `grade`, `sqft_living`, `hous_val_amt`,
`sqft_living15`, `per_bchlr` — top 5, consistente com a hipótese: tamanho/qualidade dominam, mas
demografia do zipcode (`hous_val_amt`, `per_bchlr`) carrega sinal quase tão forte quanto atributos
físicos. `condition` é o mais fraco isolado (ver tabela `corr` acima para os valores exatos desta
execução).

**Padrão geográfico:** gradiente espacial claro, não aleatório — razão de ~8x entre a mediana de preço
do zipcode mais caro e do mais barato (ver `razao` acima).

![correlação](../reports/figures/02_correlation_price_log.png)
![padrão geográfico](../reports/figures/02_geo_price_pattern.png)

## Descoberta de representação geográfica

70 zipcodes (`report["n_zipcodes"]`), `n` de imóveis por zipcode variando entre 50 e 601
(`report["zip_summary_min_n"]`/`report["zip_summary_max_n"]`) — 3 zipcodes têm menos de 100 imóveis no
total (`report["n_zipcodes_below_100_properties"]`): 98039, 98148, 98024
(`report["low_volume_zipcodes"]`). **Achado relevante:** `98039` (Medina) é simultaneamente o zipcode
**mais caro** e um dos **mais escassos** em volume de dado — é exatamente o tipo de região que um
split geográfico por zipcode isola pouco e onde erro alto é esperado. Essa observação motiva a fase 04
(`geographic_coverage`), que formaliza se escassez de dado por zipcode correlaciona com erro do
modelo — antes de decidir se data augmentation vale a pena.

![distribuição de volume por zip](../reports/figures/02_zip_volume_distribution.png)

## Validado vs. descartado

- **Validado:** estrutura espacial forte; representação desigual por zipcode é real e mensurável (não
  suposição).
- **Descartado:** nenhuma decisão de feature/modelo é tomada aqui — fase é só leitura (P1).

## Decisão

Split (fase 03) e investigação formal de cobertura geográfica (fase 04) seguem informados por esta
EDA. Nenhum artefato de modelagem gerado — só evidência.

## Artefatos

`reports/eda_summary.json`, `reports/zip_representation_summary.csv`,
`reports/figures/02_correlation_price_log.png`, `reports/figures/02_geo_price_pattern.png`,
`reports/figures/02_zip_volume_distribution.png`

# Extensão exploratória 

Abaixo: distribuição de preço (raw vs log), matriz de correlação com **todas** as features numéricas
(não só as 16 do `run_eda_pipeline`), e cruzamentos comuns de mercado imobiliário (tamanho, quartos/banheiros,
grade/condition, waterfront/view, idade do imóvel, reforma, preço por m²/zipcode). Usa o `df` já
carregado acima (`data/processed/house_clean.parquet`, 100% dos dados, pré-split). Não altera
`scripts/run_eda.py` nem os artefatos formais da fase — só gera figuras extras em `reports/figures/`.

In [7]:
from scipy import stats

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(df["price"], bins=60, color="#4C72B0")
axes[0].set_title(f"price (raw) — skew={stats.skew(df['price']):.2f}")
axes[0].set_xlabel("price")

axes[1].hist(df["price_log"], bins=60, color="#55A868")
axes[1].set_title(f"price_log — skew={stats.skew(df['price_log']):.2f}")
axes[1].set_xlabel("price_log")
plt.tight_layout()
plt.savefig("reports/figures/02_price_distribution.png", dpi=110)
plt.close()

print(df["price"].describe())
print("\nkurtosis price (raw):", stats.kurtosis(df["price"]))
print("kurtosis price_log:", stats.kurtosis(df["price_log"]))
print("\nsaved reports/figures/02_price_distribution.png")

count    2.161200e+04
mean     5.400835e+05
std      3.671351e+05
min      7.500000e+04
25%      3.218375e+05
50%      4.500000e+05
75%      6.450000e+05
max      7.700000e+06
Name: price, dtype: float64

kurtosis price (raw): 34.57598358418087
kurtosis price_log: 0.6914179319902827

saved reports/figures/02_price_distribution.png


In [8]:
import seaborn as sns

numeric_df = df.select_dtypes(include=[np.number]).drop(columns=["id", "zipcode"], errors="ignore")
full_corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(full_corr, cmap="coolwarm", center=0, vmin=-1, vmax=1, square=True,
            linewidths=0.3, cbar_kws={"shrink": 0.6}, ax=ax,
            annot=full_corr.shape[0] <= 20, fmt=".2f" if full_corr.shape[0] <= 20 else "")
ax.set_title(f"correlacao entre todas as features numericas ({full_corr.shape[0]} colunas)")
plt.tight_layout()
plt.savefig("reports/figures/02_correlation_heatmap_full.png", dpi=110)
plt.close()

full_corr_price = full_corr["price_log"].drop(["price_log", "price"], errors="ignore").sort_values(key=abs, ascending=False)
print(f"{len(full_corr_price)} features (todas as numericas, exceto id/zipcode) ordenadas por |correlacao| com price_log:\n")
print(full_corr_price.to_frame("correlation_with_price_log"))
print("\nsaved reports/figures/02_correlation_heatmap_full.png")

43 features (todas as numericas, exceto id/zipcode) ordenadas por |correlacao| com price_log:

                        correlation_with_price_log
grade                                     0.703661
sqft_living                               0.695365
hous_val_amt                              0.630817
sqft_living15                             0.619358
per_hsd                                  -0.615162
per_bchlr                                 0.611401
medn_incm_per_prsn_amt                    0.605371
sqft_above                                0.601844
per_prfsnl                                0.599427
bathrooms                                 0.550823
per_9_to_12                              -0.520585
edctn_high_schl_qty                      -0.480315
edctn_9_12_qty                           -0.470939
lat                                       0.449161
edctn_prfsnl_qty                          0.393065
bedrooms                                  0.351005
view                                  

In [9]:
fig, ax = plt.subplots(figsize=(8, 6.5))
sc = ax.scatter(df["sqft_living"], df["price"], c=df["grade"], cmap="viridis", s=8, alpha=0.4)
ax.set_title("price vs sqft_living (cor = grade)")
ax.set_xlabel("sqft_living")
ax.set_ylabel("price")
plt.colorbar(sc, ax=ax, label="grade")
plt.tight_layout()
plt.savefig("reports/figures/02_price_vs_sqft_living.png", dpi=110)
plt.close()
print("saved reports/figures/02_price_vs_sqft_living.png")

saved reports/figures/02_price_vs_sqft_living.png


In [10]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
bed_order = sorted(df.loc[df["bedrooms"].between(1, 7), "bedrooms"].unique())
sns.boxplot(data=df[df["bedrooms"].between(1, 7)], x="bedrooms", y="price", order=bed_order,
            ax=axes[0], color="#4C72B0")
axes[0].set_title("price por bedrooms (1-7)")

bath_bins = sorted(df.loc[df["bathrooms"].between(1, 4.5), "bathrooms"].unique())
sns.boxplot(data=df[df["bathrooms"].between(1, 4.5)], x="bathrooms", y="price", order=bath_bins,
            ax=axes[1], color="#DD8452")
axes[1].set_title("price por bathrooms (1-4.5)")
axes[1].tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.savefig("reports/figures/02_price_by_bedrooms_bathrooms.png", dpi=110)
plt.close()
print("saved reports/figures/02_price_by_bedrooms_bathrooms.png")

saved reports/figures/02_price_by_bedrooms_bathrooms.png


In [11]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.boxplot(data=df, x="grade", y="price", ax=axes[0], color="#55A868")
axes[0].set_title("price por grade")
axes[0].tick_params(axis="x", rotation=0)

sns.boxplot(data=df, x="condition", y="price", ax=axes[1], color="#C44E52")
axes[1].set_title("price por condition")
plt.tight_layout()
plt.savefig("reports/figures/02_price_by_grade_condition.png", dpi=110)
plt.close()
print("saved reports/figures/02_price_by_grade_condition.png")

saved reports/figures/02_price_by_grade_condition.png


In [12]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.boxplot(data=df, x="waterfront", y="price", ax=axes[0], color="#4C72B0")
axes[0].set_title("price por waterfront (0/1)")

sns.boxplot(data=df, x="view", y="price", ax=axes[1], color="#DD8452")
axes[1].set_title("price por view (0-4)")
plt.tight_layout()
plt.savefig("reports/figures/02_price_waterfront_view.png", dpi=110)
plt.close()

wf_median = df.groupby("waterfront")["price"].median()
print("mediana price por waterfront:\n", wf_median)
print("\nrazao waterfront/sem:", wf_median[1] / wf_median[0])
print("\nsaved reports/figures/02_price_waterfront_view.png")

mediana price por waterfront:
 waterfront
0     450000.0
1    1400000.0
Name: price, dtype: float64

razao waterfront/sem: 3.111111111111111

saved reports/figures/02_price_waterfront_view.png


In [13]:
df["property_age_at_sale"] = pd.to_datetime(df["date"]).dt.year - df["yr_built"]

fig, ax = plt.subplots(figsize=(9, 5.5))
age_median = df.groupby(df["property_age_at_sale"].clip(upper=100))["price"].median()
ax.plot(age_median.index, age_median.values, color="#4C72B0")
ax.set_title("mediana de price por idade do imovel na venda (clip 100 anos)")
ax.set_xlabel("idade (anos)")
ax.set_ylabel("mediana price")
plt.tight_layout()
plt.savefig("reports/figures/02_price_vs_age.png", dpi=110)
plt.close()

print("correlacao property_age_at_sale x price_log:", df["property_age_at_sale"].corr(df["price_log"]))
print("saved reports/figures/02_price_vs_age.png")

correlacao property_age_at_sale x price_log: -0.08059895554518534
saved reports/figures/02_price_vs_age.png


In [14]:
df["was_renovated"] = (df["yr_renovated"] > 0).astype(int)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
sns.boxplot(data=df, x="was_renovated", y="price", ax=ax, color="#55A868")
ax.set_xticklabels(["nao reformado", "reformado"])
ax.set_title("price: reformado vs nao reformado")
plt.tight_layout()
plt.savefig("reports/figures/02_price_renovated.png", dpi=110)
plt.close()

renov_median = df.groupby("was_renovated")["price"].median()
print("mediana price:\n", renov_median)
print("\nrazao reformado/nao:", renov_median[1] / renov_median[0])
print("% reformados no dataset:", df["was_renovated"].mean() * 100)
print("\nsaved reports/figures/02_price_renovated.png")

mediana price:
 was_renovated
0    448000.0
1    600000.0
Name: price, dtype: float64

razao reformado/nao: 1.3392857142857142
% reformados no dataset: 4.229131963723857

saved reports/figures/02_price_renovated.png


C:\Users\br3nd\AppData\Local\Temp\ipykernel_4180\3489231474.py:5: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator. Otherwise, ticks may be mislabeled.
  ax.set_xticklabels(["nao reformado", "reformado"])


In [15]:
df["price_per_sqft"] = df["price"] / df["sqft_living"]
zip_pps = df.groupby("zipcode")["price_per_sqft"].median().sort_values()

fig, ax = plt.subplots(figsize=(8, 12))
ax.barh(zip_pps.index.astype(str), zip_pps.values, color="#4C72B0")
ax.set_title("mediana price/sqft_living por zipcode")
ax.tick_params(axis="y", labelsize=5)
plt.tight_layout()
plt.savefig("reports/figures/02_price_per_sqft_by_zipcode.png", dpi=110)
plt.close()

print("price/sqft_living, top 5 mais caros:\n", zip_pps.tail(5))
print("\nprice/sqft_living, top 5 mais baratos:\n", zip_pps.head(5))
print("\nsaved reports/figures/02_price_per_sqft_by_zipcode.png")

price/sqft_living, top 5 mais caros:
 zipcode
98119    416.652778
98112    423.828125
98109    427.696078
98004    456.944444
98039    565.165614
Name: price_per_sqft, dtype: float64

price/sqft_living, top 5 mais baratos:
 zipcode
98023    144.970414
98032    146.205357
98002    147.177419
98001    147.275708
98092    149.180328
Name: price_per_sqft, dtype: float64

saved reports/figures/02_price_per_sqft_by_zipcode.png


## Notas sobre a extensão exploratória

- **Distribuição:** `price` bruto é fortemente assimétrico à direita (skew alto); `price_log` reduz a
  assimetria — reforça por que o pipeline modela sobre `price_log`.
- **Correlação completa:** o heatmap com todas as colunas numéricas mostra colinearidade forte dentro
  dos blocos de tamanho (`sqft_living`/`sqft_above`/`sqft_living15`) e dentro do bloco demográfico
  (`hous_val_amt`/`medn_hshld_incm_amt`/`per_bchlr`/`edctn_*`) — útil para decisão futura de seleção de
  features (fases 05-09), não decidido aqui (fase é só leitura, P1).
- **Cruzamentos:** `grade`, `sqft_living`, `bathrooms` têm relação monotônica clara com `price`;
  `waterfront` e `view` têm poucos imóveis mas prêmio de preço visível; efeito de reforma e idade do
  imóvel são mais ruidosos, olhando o preço bruto sem controlar por outras variáveis (ver `razao` e
  `correlacao` impressos em cada célula acima).
- Nenhum artefato desta extensão é lido por `scripts/run_eda.py` ou por fases posteriores — figuras
  ficam em `reports/figures/02_price_*`, `02_correlation_heatmap_full.png` e
  `02_price_per_sqft_by_zipcode.png`, só para leitura exploratória.